# Composition Stratification Analysis

**Goals:**
1. Characterise the MP training set vs WBM test set by composition complexity
2. Quantify the coverage gap — which chemistries are underrepresented in training?
3. Design a reduced but representative training subset for fast hyperparameter experiments
4. Design a composition-stratified oversampling strategy for full training

**Definition of 'similar variance':**
- Same distribution of n_unique_elements
- Same distribution of formation energy values (label range)
- Same distribution of n_sites
- Similar element-type coverage

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from collections import Counter
from pymatgen.core import Composition

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 120,
})
BLUE = '#4C72B0'; ORANGE = '#DD8452'; GREEN = '#55A868'; RED = '#C44E52'; GREY = '#8C8C8C'

REPO_ROOT = Path('..').resolve()
OUT_DIR   = REPO_ROOT / 'results' / 'stratification'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Build data cache if not already present
import subprocess, sys
from pathlib import Path

cache_dir = REPO_ROOT / 'data' / 'processed'
train_pt  = cache_dir / 'train.pt'

if train_pt.exists():
    print('Cache already exists — skipping build.')
else:
    print('Building cache (~20-40 min on CPU)...')
    result = subprocess.run(
        [sys.executable, str(REPO_ROOT / 'scripts' / 'build_cache.py')]
    )
    print(f'Done. Exit code: {result.returncode}')

Building cache (~20-40 min on CPU)...


## Section 1 — Load MP training data

In [2]:
# Load cached MP train+test data
data_dir   = REPO_ROOT / 'data' / 'processed'
train_data = torch.load(data_dir / 'train.pt', weights_only=False)
test_data  = torch.load(data_dir / 'test.pt',  weights_only=False)
all_data   = train_data + test_data

print(f'MP train : {len(train_data):,}')
print(f'MP test  : {len(test_data):,}')
print(f'MP total : {len(all_data):,}')

# Extract per-structure features from PyG Data objects
mp_records = []
for d in all_data:
    n_sites    = d.x.shape[0]                          # number of atoms
    n_unique   = int(d.x.sum(dim=0).gt(0).sum().item()) # non-zero one-hot cols = unique elements
    e_form     = float(d.y.item())
    mp_records.append({'n_sites': n_sites, 'n_unique_elements': n_unique, 'e_form': e_form})

mp_df = pd.DataFrame(mp_records)
print(f'\nMP dataset summary:')
display(mp_df.describe().round(3))

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Projects\\gnome-repro-structural\\data\\processed\\train.pt'

## Section 2 — Load WBM summary

In [ ]:
wbm = pd.read_csv(REPO_ROOT / 'data' / 'raw' / 'wbm-summary.csv.gz')

def n_unique_elements(f):
    try: return len(Composition(f).elements)
    except: return np.nan

wbm['n_unique_elements'] = wbm['formula'].apply(n_unique_elements)
wbm['true_stable']       = wbm['e_above_hull_wbm'] <= 0.0

print(f'WBM structures: {len(wbm):,}')
display(wbm[['n_sites','n_unique_elements','e_form_per_atom_wbm','e_above_hull_wbm']].describe().round(3))

## Section 3 — Distribution comparison: MP vs WBM

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# n_unique_elements
ax = axes[0]
el_range = range(1, 8)
mp_el_counts  = [( mp_df['n_unique_elements'] == n).sum() for n in el_range]
wbm_el_counts = [(wbm['n_unique_elements']    == n).sum() for n in el_range]
mp_el_frac    = np.array(mp_el_counts)  / len(mp_df)  * 100
wbm_el_frac   = np.array(wbm_el_counts) / len(wbm)    * 100
x = np.array(list(el_range))
ax.bar(x - 0.2, mp_el_frac,  0.4, color=BLUE,   alpha=0.75, label='MP train')
ax.bar(x + 0.2, wbm_el_frac, 0.4, color=ORANGE, alpha=0.75, label='WBM test')
ax.set_xlabel('Unique elements'); ax.set_ylabel('% of dataset')
ax.set_title('Chemical complexity distribution')
ax.legend(); ax.set_xticks(list(el_range))

# n_sites
ax = axes[1]
bins = [1, 5, 10, 20, 40, 80, 200]
labels = ['1-4','5-9','10-19','20-39','40-79','80+']
mp_sites  = pd.cut(mp_df['n_sites'], bins=bins, labels=labels, right=False).value_counts(sort=False)
wbm_sites = pd.cut(wbm['n_sites'],   bins=bins, labels=labels, right=False).value_counts(sort=False)
mp_sites_pct  = mp_sites  / len(mp_df)  * 100
wbm_sites_pct = wbm_sites / len(wbm)    * 100
x2 = np.arange(len(labels))
ax.bar(x2 - 0.2, mp_sites_pct.values,  0.4, color=BLUE,   alpha=0.75, label='MP train')
ax.bar(x2 + 0.2, wbm_sites_pct.values, 0.4, color=ORANGE, alpha=0.75, label='WBM test')
ax.set_xticks(x2); ax.set_xticklabels(labels, rotation=30)
ax.set_xlabel('N sites'); ax.set_ylabel('% of dataset')
ax.set_title('Structure size distribution'); ax.legend()

# Formation energy
ax = axes[2]
ax.hist(mp_df['e_form'],              bins=80, range=(-4,1), color=BLUE,   alpha=0.6,
        density=True, label='MP train')
ax.hist(wbm['e_form_per_atom_wbm'],   bins=80, range=(-4,1), color=ORANGE, alpha=0.6,
        density=True, label='WBM test')
ax.set_xlabel('Formation energy (eV/atom)'); ax.set_ylabel('Density')
ax.set_title('Formation energy distribution'); ax.legend()

fig.suptitle('MP training set vs WBM test set', fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR / '01_mp_vs_wbm_distributions.png', dpi=150)
plt.show()
print('Saved: 01_mp_vs_wbm_distributions.png')

In [ ]:
# Coverage gap table
print('Coverage gap: MP vs WBM by n_unique_elements')
rows = []
for n in el_range:
    mp_n  = (mp_df['n_unique_elements'] == n).sum()
    wbm_n = (wbm['n_unique_elements']   == n).sum()
    rows.append({
        'n_elements'   : n,
        'MP_count'     : mp_n,
        'MP_%'         : round(100 * mp_n  / len(mp_df), 2),
        'WBM_count'    : wbm_n,
        'WBM_%'        : round(100 * wbm_n / len(wbm),   2),
        'coverage_ratio': round(mp_n / max(wbm_n, 1), 3),
    })
gap_df = pd.DataFrame(rows)
display(gap_df)
gap_df.to_csv(OUT_DIR / '02_coverage_gap.csv', index=False)
print('Saved: 02_coverage_gap.csv')
print()
print('coverage_ratio < 1 means WBM has MORE structures of that type than MP training.')
print('This is the extrapolation zone where the model will struggle most.')

## Section 4 — Design reduced dataset for hyperparameter experiments

In [ ]:
# Target: ~20% of full MP dataset, stratified by n_unique_elements
# Each element-count bin gets proportional representation
# BUT we oversample quaternaries/quinaries to match WBM distribution

TARGET_SIZE   = int(len(mp_df) * 0.20)   # ~20% of full dataset
OVERSAMPLE_EL = {4: 2.0, 5: 3.0}         # oversample 4-element by 2x, 5-element by 3x

print(f'Full MP dataset    : {len(mp_df):,}')
print(f'Target reduced size: {TARGET_SIZE:,} (~20%)')
print()

# Compute target counts per element group
base_frac = mp_el_frac / 100   # natural frequency
adjusted_frac = base_frac.copy()
for i, n_el in enumerate(el_range):
    if n_el in OVERSAMPLE_EL:
        adjusted_frac[i] *= OVERSAMPLE_EL[n_el]
adjusted_frac /= adjusted_frac.sum()   # renormalise

print('Sampling plan for reduced dataset:')
print(f'{"n_el":>6} {"natural_%":>12} {"adjusted_%":>12} {"target_n":>10} {"available_n":>12}')
for i, n_el in enumerate(el_range):
    target_n    = int(adjusted_frac[i] * TARGET_SIZE)
    available_n = mp_el_counts[i]
    flag = '  *** OVERSAMPLE' if n_el in OVERSAMPLE_EL else ''
    print(f'{n_el:>6} {base_frac[i]*100:>11.1f}% {adjusted_frac[i]*100:>11.1f}%'
          f' {target_n:>10,} {available_n:>12,}{flag}')

In [ ]:
# Build the reduced dataset indices
mp_df_indexed = mp_df.copy().reset_index()   # preserve original index
reduced_indices = []
rng = np.random.default_rng(seed=42)

for i, n_el in enumerate(el_range):
    target_n    = int(adjusted_frac[i] * TARGET_SIZE)
    available   = mp_df_indexed[mp_df_indexed['n_unique_elements'] == n_el]['index'].values
    if len(available) == 0:
        continue
    chosen = rng.choice(available, size=min(target_n, len(available)), replace=False)
    reduced_indices.extend(chosen.tolist())

reduced_df = mp_df.iloc[reduced_indices].copy()
print(f'\nReduced dataset: {len(reduced_df):,} structures')
print('Element distribution in reduced set:')
for n_el in el_range:
    n = (reduced_df['n_unique_elements'] == n_el).sum()
    print(f'  {n_el} elements: {n:,} ({100*n/len(reduced_df):.1f}%)')

# Save indices for use in training
np.save(OUT_DIR / 'reduced_dataset_indices.npy', np.array(reduced_indices))
print(f'\nIndices saved to: {OUT_DIR}/reduced_dataset_indices.npy')

In [ ]:
# Verify reduced dataset has similar variance to full dataset
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# n_unique_elements
ax = axes[0]
full_el    = [(mp_df['n_unique_elements'] == n).sum() / len(mp_df) * 100  for n in el_range]
reduced_el = [(reduced_df['n_unique_elements'] == n).sum() / len(reduced_df) * 100 for n in el_range]
wbm_el_    = [(wbm['n_unique_elements'] == n).sum() / len(wbm) * 100 for n in el_range]
x = np.array(list(el_range))
ax.bar(x - 0.25, full_el,    0.25, color=BLUE,   alpha=0.75, label='MP full')
ax.bar(x,        reduced_el, 0.25, color=GREEN,  alpha=0.75, label='MP reduced')
ax.bar(x + 0.25, wbm_el_,   0.25, color=ORANGE, alpha=0.75, label='WBM')
ax.set_xticks(list(el_range))
ax.set_xlabel('Unique elements'); ax.set_ylabel('%')
ax.set_title('Element complexity'); ax.legend(fontsize=8)

# Formation energy
ax = axes[1]
ax.hist(mp_df['e_form'],         bins=60, range=(-4,1), color=BLUE,  alpha=0.5, density=True, label='MP full')
ax.hist(reduced_df['e_form'],    bins=60, range=(-4,1), color=GREEN, alpha=0.6, density=True, label='MP reduced')
ax.set_xlabel('Formation energy (eV/atom)'); ax.set_ylabel('Density')
ax.set_title('Label distribution'); ax.legend(fontsize=8)

# n_sites
ax = axes[2]
bins_s = [1,5,10,20,40,80,200]; labels_s = ['1-4','5-9','10-19','20-39','40-79','80+']
full_s    = pd.cut(mp_df['n_sites'],      bins=bins_s, labels=labels_s, right=False).value_counts(sort=False) / len(mp_df) * 100
reduced_s = pd.cut(reduced_df['n_sites'], bins=bins_s, labels=labels_s, right=False).value_counts(sort=False) / len(reduced_df) * 100
x2 = np.arange(len(labels_s))
ax.bar(x2 - 0.2, full_s.values,    0.4, color=BLUE,  alpha=0.75, label='MP full')
ax.bar(x2 + 0.2, reduced_s.values, 0.4, color=GREEN, alpha=0.75, label='MP reduced')
ax.set_xticks(x2); ax.set_xticklabels(labels_s, rotation=30)
ax.set_xlabel('N sites'); ax.set_ylabel('%')
ax.set_title('Size distribution'); ax.legend(fontsize=8)

fig.suptitle('Full MP vs Reduced dataset vs WBM', fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR / '03_reduced_dataset_check.png', dpi=150)
plt.show()
print('Saved: 03_reduced_dataset_check.png')

## Section 5 — Oversampling strategy for full training

In [ ]:
# How many quaternaries/quinaries would we need to match WBM proportions?
print('Oversampling analysis — matching WBM element distribution in MP training')
print()
print('Strategy: for each element group, compute how many times we need to')
print('repeat MP samples to match the WBM proportion.')
print()

total_mp = len(mp_df)
rows_os = []
for n_el in el_range:
    mp_n    = (mp_df['n_unique_elements'] == n_el).sum()
    wbm_n   = (wbm['n_unique_elements']   == n_el).sum()
    mp_frac = mp_n  / total_mp
    wbm_frac= wbm_n / len(wbm)
    # How many times do we need to repeat MP samples of this type
    # to achieve the same fraction as WBM?
    if mp_n > 0:
        required_n      = int(wbm_frac * total_mp)
        oversample_factor = round(required_n / mp_n, 2)
    else:
        required_n = 0; oversample_factor = 0

    rows_os.append({
        'n_elements'        : n_el,
        'MP_count'          : mp_n,
        'MP_fraction_%'     : round(mp_frac*100, 2),
        'WBM_fraction_%'    : round(wbm_frac*100, 2),
        'required_n'        : required_n,
        'oversample_factor' : oversample_factor,
    })

os_df = pd.DataFrame(rows_os)
display(os_df)
os_df.to_csv(OUT_DIR / '04_oversampling_strategy.csv', index=False)
print('Saved: 04_oversampling_strategy.csv')
print()
print('oversample_factor > 1 means we need to repeat those samples.')
print('oversample_factor >> 1 means MP simply does not have enough of that type.')
print('For those, MPtrj early frames or WBM pairs are the only real fix.')

## Section 6 — Summary and recommendations

In [ ]:
print('=== STRATIFICATION SUMMARY ===')
print()
print('1. REDUCED DATASET')
print(f'   Size         : {len(reduced_df):,} structures (~20% of full MP)')
print(f'   Indices saved: results/stratification/reduced_dataset_indices.npy')
print(f'   Use for      : fast hyperparameter experiments (dropout, weight decay,')
print(f'                  n_layers, Huber loss) before committing to full retrain')
print()
print('2. OVERSAMPLING FOR FULL TRAINING')
print('   Oversample quaternaries (4-el) by ~2x')
print('   Oversample quinaries (5-el) by ~3x')
print('   This improves WBM alignment without new data')
print()
print('3. DATA AUGMENTATION (NEXT STEP)')
print('   MPtrj early frames (unrelaxed-like) address Root Cause 1 directly')
print('   WBM initial+relaxed pairs address Root Cause 1 even more directly')
print('   Both are available as open datasets from Matbench Discovery')